In [ ]:
# Install Google Chrome + ChromeDriver for Colab
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -qq
!pip install webdriver-manager selenium imagehash Pillow requests -q
!google-chrome --version
print('Chrome installed!')

Selecting previously unselected package libatk1.0-data.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../02-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libatk-bridge2.0-0:amd64.
Preparing to unpack .../03-libatk-bridge2.0-0_2.38.0-3_amd64.deb ...
Unpacking libatk-bridge2.0-0:amd64 (2.38.0-3) ...
Selecting previously unselected package libvulkan1:amd64.
Preparing to unpack .../04-libvulkan1_1.3.204.1-2_amd64.deb ...
Unpacking libvulkan1:amd64 (1.3.204.1-2) ...
Selecting previously unselected package libxcomposite1:amd

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/Project2_CNN'
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
print('Working dir:', os.getcwd())

Mounted at /content/drive
Working dir: /content/drive/MyDrive/Project2_CNN


In [ ]:
# Import all libraries
import os, shutil, random, time, requests, re, warnings, pickle
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from io import BytesIO
import imagehash
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, GlobalAveragePooling2D, BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, EfficientNetB0, VGG16, ResNet101
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

np.random.seed(42)
tf.random.set_seed(42)
print('Libraries ready!')
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

Libraries ready!
TF: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 30
PATIENCE    = 7
NUM_CLASSES = 16
POLITICIANS = [
    'imran_khan', 'nawaz_sharif', 'shehbaz_sharif',
    'asif_ali_zardari', 'bilawal_bhutto', 'maryam_nawaz',
    'fazlur_rehman', 'pervez_elahi', 'aitzaz_ahsan',
    'khurshid_shah', 'ishaq_dar', 'chaudhry_nisar',
    'shah_mahmood_qureshi', 'asad_umar',
    'rana_sanaullah', 'ahmed_sharif_chaudhry'
]
WORK_DIR    = os.getcwd()
DATASET_DIR = os.path.join(WORK_DIR, 'dataset_final')
TRAIN_DIR   = os.path.join(DATASET_DIR, 'train')
VAL_DIR     = os.path.join(DATASET_DIR, 'val')
TEST_DIR    = os.path.join(DATASET_DIR, 'test')
RAW_DIR     = os.path.join(WORK_DIR, 'raw_downloads')
CLEAN_DIR   = os.path.join(WORK_DIR, 'clean_images')
print(f'Project dir: {WORK_DIR}')
print(f'Classes    : {NUM_CLASSES}')

Project dir: /content/drive/MyDrive/Project2_CNN
Classes    : 16


In [ ]:
SEARCH_QUERIES = {
    'imran_khan'           : 'Imran Khan PTI Pakistan face photo',
    'nawaz_sharif'         : 'Nawaz Sharif PMLN Pakistan face photo',
    'shehbaz_sharif'       : 'Shehbaz Sharif PM Pakistan face photo',
    'asif_ali_zardari'     : 'Asif Zardari PPP Pakistan face photo',
    'bilawal_bhutto'       : 'Bilawal Bhutto PPP Pakistan face photo',
    'maryam_nawaz'         : 'Maryam Nawaz Punjab CM face photo',
    'fazlur_rehman'        : 'Fazlur Rehman JUI Pakistan face photo',
    'pervez_elahi'         : 'Pervez Elahi politician Pakistan face photo',
    'aitzaz_ahsan'         : 'Aitzaz Ahsan lawyer Pakistan face photo',
    'khurshid_shah'        : 'Khurshid Shah PPP Pakistan face photo',
    'ishaq_dar'            : 'Ishaq Dar finance minister Pakistan face photo',
    'chaudhry_nisar'       : 'Chaudhry Nisar politician Pakistan face photo',
    'shah_mahmood_qureshi' : 'Shah Mahmood Qureshi PTI face photo',
    'asad_umar'            : 'Asad Umar PTI Pakistan face photo',
    'rana_sanaullah'       : 'Rana Sanaullah interior minister Pakistan face photo',
    'ahmed_sharif_chaudhry': 'Ahmed Sharif Chaudhry ISPR Pakistan face photo',
}
IMAGES_TARGET  = 100
HASH_THRESHOLD = 8
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}
print('Queries ready!')

Queries ready!


In [ ]:
from webdriver_manager.chrome import ChromeDriverManager
def create_driver():
    opt = Options()
    opt.add_argument('--headless=new')
    opt.add_argument('--no-sandbox')
    opt.add_argument('--disable-dev-shm-usage')
    opt.add_argument('--disable-gpu')
    opt.add_argument('--window-size=1280,900')
    opt.add_argument('--disable-blink-features=AutomationControlled')
    opt.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
    )
    opt.add_experimental_option('excludeSwitches', ['enable-automation'])
    # Auto download matching chromedriver
    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=opt)
    return driver
# Test
print('Testing Chrome...')
d = create_driver()
d.get('https://www.google.com')
print('OK! Title:', d.title)
d.quit()
print('Chrome ready!')

Testing Chrome...
OK! Title: Google
Chrome ready!


In [ ]:
def extract_urls(page_source):
    pattern = r'"(https?://(?!encrypted)[^"]+\.(?:jpg|jpeg|png)(?:\?[^"]*)?)"'
    urls    = re.findall(pattern, page_source)
    skip    = ['gstatic','google.com/images','googleusercontent.com/proxy']
    seen, result = set(), []
    for url in urls:
        if any(s in url for s in skip): continue
        if len(url) < 30: continue
        if url not in seen:
            seen.add(url)
            result.append(url)
    return result
def scrape_google(driver, query, max_urls=100):
    url = f'https://www.google.com/search?q={query.replace(" ","+")}&tbm=isch&tbs=itp:photo&hl=en'
    driver.get(url); time.sleep(3)
    for text in ['Accept all','Accept','I agree']:
        try:
            driver.find_element(By.XPATH, f'//button[contains(.,"{text}")]').click()
            time.sleep(1); break
        except: pass
    all_urls = set()
    for _ in range(8):
        if len(all_urls) >= max_urls*2: break
        all_urls.update(extract_urls(driver.page_source))
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(2)
    return list(all_urls)[:max_urls]
def scrape_bing(driver, query, max_urls=100):
    url = f'https://www.bing.com/images/search?q={query.replace(" ","+")}&qft=+filterui:photo-photo'
    driver.get(url); time.sleep(3)
    all_urls = set()
    for _ in range(8):
        if len(all_urls) >= max_urls*2: break
        all_urls.update(extract_urls(driver.page_source))
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(2)
    return list(all_urls)[:max_urls]
def download_urls(urls, folder, prefix, max_num=100):
    save_dir = os.path.join(RAW_DIR, folder)
    os.makedirs(save_dir, exist_ok=True)
    downloaded = 0
    for i, url in enumerate(urls):
        if downloaded >= max_num: break
        try:
            resp = requests.get(url, timeout=6, headers=HEADERS)
            if resp.status_code != 200: continue
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            if img.width < 80 or img.height < 80: continue
            img.save(os.path.join(save_dir, f'{prefix}_{i:04d}.jpg'), 'JPEG', quality=90)
            downloaded += 1
        except: continue
    return downloaded
def remove_duplicates(folder):
    raw_folder   = os.path.join(RAW_DIR, folder)
    clean_folder = os.path.join(CLEAN_DIR, folder)
    os.makedirs(clean_folder, exist_ok=True)
    if not os.path.exists(raw_folder): return 0
    images = [os.path.join(raw_folder, f)
              for f in os.listdir(raw_folder)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]
    seen, saved = [], 0
    for p in images:
        try:
            img = Image.open(p).convert('RGB')
            if img.width < 80 or img.height < 80: continue
            h = imagehash.phash(img)
            if any(abs(h-sh) <= HASH_THRESHOLD for sh in seen): continue
            seen.append(h)
            img.save(os.path.join(clean_folder, f'{folder}_{saved:04d}.jpg'), 'JPEG', quality=90)
            saved += 1
        except: continue
    return saved

def split_dataset(folder):
    clean_folder = os.path.join(CLEAN_DIR, folder)
    if not os.path.exists(clean_folder): return 0,0,0
    images = [f for f in os.listdir(clean_folder) if f.endswith('.jpg')]
    random.shuffle(images)
    total = len(images)
    t_end = int(total*0.75); v_end = int(total*0.90)
    splits = {'train':images[:t_end],'val':images[t_end:v_end],'test':images[v_end:]}
    for name, files in splits.items():
        dest = os.path.join(DATASET_DIR, name, folder)
        os.makedirs(dest, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(clean_folder,f), os.path.join(dest,f))
    return len(splits['train']), len(splits['val']), len(splits['test'])

print('Helper functions ready!')

Helper functions ready!


In [ ]:
print('Starting Chromium (single window)...')
chrome_driver = create_driver()
print('Ready! Starting collection...\n')
summary = []
try:
    for i, (folder, query) in enumerate(SEARCH_QUERIES.items()):
        # Skip if already collected
        clean_folder = os.path.join(CLEAN_DIR, folder)
        if os.path.exists(clean_folder) and len(os.listdir(clean_folder)) >= 80:
            print(f'[{i+1}/16] {folder} — already done, skipping')
            t,v,te = split_dataset(folder)
            summary.append({'folder':folder,'total':t+v+te,'status':'OK'})
            continue
        print(f'\n[{i+1}/16] {folder}')
        # Google
        g_urls = scrape_google(chrome_driver, query, max_urls=100)
        g_dl   = download_urls(g_urls, folder, f'{folder}_g', max_num=100)
        print(f'  Google : {g_dl}')
        time.sleep(3)
        # Bing
        b_urls = scrape_bing(chrome_driver, query, max_urls=100)
        b_dl   = download_urls(b_urls, folder, f'{folder}_b', max_num=100)
        print(f'  Bing   : {b_dl}')
        # Clean
        clean = remove_duplicates(folder)
        print(f'  Clean  : {clean}')
        # Split
        t,v,te = split_dataset(folder)
        print(f'  Split  : Train:{t} Val:{v} Test:{te}')
        status = 'OK' if (t+v+te)>=80 else 'NEED MORE'
        summary.append({'folder':folder,'total':t+v+te,'status':status})
        time.sleep(5)

finally:
    chrome_driver.quit()
    print('\nChromium closed.')
print('COLLECTION SUMMARY')

for s in summary:
    mk = 'OK' if s['status']=='OK' else 'NEED MORE'
    print(f"{s['folder']:<28} {s['total']:>4}  {mk}")
print('='*52)
print(f"Total: {sum(s['total'] for s in summary)} images")

Starting Chromium (single window)...
Ready! Starting collection...


[1/16] imran_khan
  Google : 65
  Bing   : 1
  Clean  : 66
  Split  : Train:49 Val:10 Test:7

[2/16] nawaz_sharif
  Google : 70
  Bing   : 1
  Clean  : 71
  Split  : Train:53 Val:10 Test:8

[3/16] shehbaz_sharif
  Google : 66
  Bing   : 1
  Clean  : 67
  Split  : Train:50 Val:10 Test:7

[4/16] asif_ali_zardari
  Google : 72
  Bing   : 1
  Clean  : 72
  Split  : Train:54 Val:10 Test:8

[5/16] bilawal_bhutto
  Google : 59
  Bing   : 1
  Clean  : 60
  Split  : Train:45 Val:9 Test:6

[6/16] maryam_nawaz
  Google : 62
  Bing   : 1
  Clean  : 63
  Split  : Train:47 Val:9 Test:7

[7/16] fazlur_rehman
  Google : 72
  Bing   : 1
  Clean  : 73
  Split  : Train:54 Val:11 Test:8

[8/16] pervez_elahi
  Google : 71
  Bing   : 1
  Clean  : 71
  Split  : Train:53 Val:10 Test:8

[9/16] aitzaz_ahsan
  Google : 61
  Bing   : 1
  Clean  : 62
  Split  : Train:46 Val:9 Test:7

[10/16] khurshid_shah
  Google : 70
  Bing   : 1
  Clean  : 71
